## Local RAG Pipeline Using FAISS + Mistral-7B (via llama-cpp-python)

**Step 1: Import Required Libraries**

+ llama_cpp.Llama is used to load and interact with your local Mistral 7B GGUF model.
+ This setup allows you to run a powerful language model locally, even offline.


In [1]:
# For semantic similarity search over embeddings
import faiss
# For reading and manipulating the chunked CSV
import pandas as pd
# For encoding queries into vector form
from sentence_transformers import SentenceTransformer
# For numerical operations
import numpy as np
# llama-cpp-python interface for Mistral    
from llama_cpp import Llama  

c:\Users\brian\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


**Step 2: Load FAISS Index and Chunked CSV**
+ The FAISS index contains vector embeddings of your PaddleOCR-generated text chunks.
+ The CSV provides the actual content (text) associated with each embedding.

In [2]:
# Load FAISS index and chunked text 
faiss_path = r"C:\Users\brian\OneDrive\Escritorio\Skills\Programming\Python\Project\faiss_index_PaddleOCR.index"
csv_path = r"C:\Users\brian\OneDrive\Escritorio\Skills\Programming\Python\Project\extracted_text_PaddleOCR2\cleaned_text_PaddleOCR2\chunked_text_PaddleOCR.csv"

index = faiss.read_index(faiss_path)
df_chunks = pd.read_csv(csv_path)

**Step 3: Load the Sentence Embedding Model**
+ We use the "all-MiniLM-L6-v2" model to encode both your text chunks and queries.
+ This ensures that semantic comparisons use the same embedding space.

In [3]:
# Load SentenceTransformer for embeddings 
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

**Step 4: Define Function to Retrieve Top-k Chunks**

This function:
+ Converts the user’s query to an embedding.
+ Finds the top k most similar text chunks.
+ Returns a subset of the dataframe with the most relevant rows.



In [4]:
# Step 3: Define search function
def search_top_k(query, k=2):
    query_embedding = embedding_model.encode([query])
    distances, indices = index.search(np.array(query_embedding).astype("float32"), k)
    return df_chunks.iloc[indices[0]][["chunk_id", "filename", "chunk_text"]]

**Step 5: Load Mistral-7B Locally Using llama-cpp**
+ Loads a quantized .gguf model of Mistral-7B using llama-cpp-python.
+ n_ctx=2048 sets the context window to 2048 tokens — enough for long documents.
+ Make sure the model fits your CPU/GPU memory (this example uses Q4_K_M, a 4-bit quantization).

In [5]:
# Load Mistral-7B model (via llama-cpp-python)
mistral_path = r"C:\llama_cpp\llama-b5478-bin-win-cpu-x64\mistral-7b-instruct-v0.1.Q4_K_M.gguf"
llm = Llama(model_path=mistral_path, n_ctx=2048)

llama_model_loader: loaded meta data with 20 key-value pairs and 291 tensors from C:\llama_cpp\llama-b5478-bin-win-cpu-x64\mistral-7b-instruct-v0.1.Q4_K_M.gguf (version GGUF V2)
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = mistralai_mistral-7b-instruct-v0.1
llama_model_loader: - kv   2:                       llama.context_length u32              = 32768
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 14336
llama_model_loader: - kv   6:                 llama.rope.dimension_count u32              = 128
llama_model_loa

**Step 6: Build Prompt for Local Model**

+ I create a structured prompt that guides the LLM to:
    + Use the context
    + Answer the specific question

+ This format is well-suited for instruction-tuned models like Mistral.

In [6]:
# Build prompt
def build_prompt(chunks, question):
    context = "\n".join(chunks["chunk_text"].tolist())
    prompt = (
        "You are a helpful assistant. Use the information in the context below to answer the user's question.\n\n"
        "### Context ###\n"
        f"{context}\n\n"
        "### Question ###\n"
        f"{question}\n\n"
        "### Answer ###\n"
    )
    return prompt

**Step 7: Generate Answer with Mistral-7B**
+ llm(...) generates a response using the prompt.
+ stop tokens ensure the generation halts after a complete answer.
+ The output is cleaned by removing unnecessary whitespace and stopping characters.

In [7]:
# Generate answer using Mistral
def generate_answer_mistral(chunks, question, max_tokens=200):
    prompt = build_prompt(chunks, question)
    response = llm(prompt, max_tokens=max_tokens, stop=["</s>", "###"])
    return response["choices"][0]["text"].strip()

**Step 8: Test with an Example Query**
+ Runs the full retrieval + generation pipeline on your query.
+ Retrieves relevant document chunks and generates a grounded answer from them.

In [8]:
# Test with a question 
question = "Which major event did the Indians win on August 31?"
top_chunks = search_top_k(question)
response = generate_answer_mistral(top_chunks, question)

llama_perf_context_print:        load time =   54406.03 ms
llama_perf_context_print: prompt eval time =   54405.09 ms /  1175 tokens (   46.30 ms per token,    21.60 tokens per second)
llama_perf_context_print:        eval time =    8849.90 ms /    33 runs   (  268.18 ms per token,     3.73 tokens per second)
llama_perf_context_print:       total time =   63276.78 ms /  1208 tokens


**Step 9: Show the Answer and Used Context**
+ Displays the generated response.
+ Shows the chunks that were used to answer the question — useful for debugging, transparency, or user feedback.

In [9]:
print("💬 Answer:", response)


💬 Answer: On August 31, the Indians won a 19-inning game against the Toronto Blue Jays, extending their winning streak to 14.


In [10]:
print("\n📄 Context used:")
print(top_chunks["chunk_text"].tolist())


📄 Context used:
['[ edit ] 2016 was also the high point for the Indians . The 2017 Indians won 22 straight games - the second-longest . winning streak in MLB history-on their way to a 102-60 record and AL Central title.During their 22 game', "[ 117 . ^ `` Ubaldo Jimenez traded to Indians for four players '' . USA Today . Archived from the original on August 4 2014.Retrieved September 14,2014 be named '' .cleveland.com.Archivedfrom the original on November 7,2012.Retrieved January 23,2013 119 . ^ Northeast Ohio ( September 25 , 2011 ) . `` Minnesota Twins beat Cleveland Indians , 6-4 , in 10 innings ; Jim January 23 , 2013 . 120 . ^ `` Most Popular E-mail Newsletter '' . USA Today . April 3 , 2012 . 121 . ^ `` Blue Jays Win Longest Opening Day Game In History | WBAL Home -WBAL Home '' . Wbaltv.com . April 6 , 2012.Archived from the originalon February 9,2013.Retrieved January 23 , 2013 . January 2 , 2013.Retrieved March 31 , 2013 . 123.^ '' Francona Hired as Indians manager '' .Fox Spo